# Syracuse Map & Factorization Exploration

The Syracuse map S(n) = (3n+1) / 2^v₂(3n+1) strips the Collatz function down to its odd-to-odd transitions. By removing the "noise" of even steps, we can study the multiplicative structure of orbits more clearly.

**Key question:** Does the Syracuse behavior of a composite number n = p × q relate to the Syracuse behavior of its prime factors?

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from collatz.core import (
    syracuse_step, syracuse_orbit, syracuse_stopping_time,
    v2, alpha_value, alpha_sequence,
    orbit, total_stopping_time, collatz_step,
)
from collatz.factorization import prime_factorization, is_prime
from collatz.dropping import dropping_set, orbital_oddity
from collatz.stopping import stopping_time, stopping_class

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

print("All imports successful.")

## Syracuse Basics

The Syracuse map sends odd numbers directly to odd numbers. The **alpha value** α(n) = v₂(3n+1) records how many halvings occur at each step. The alpha sequence completely determines the orbit structure.

**Key identity:** total_stopping_time(n) = len(alpha_sequence) + sum(alpha_sequence)
(each Syracuse step = 1 odd step + α even steps)

In [ ]:
# Syracuse basics for n = 3, 7, 27
for n in [3, 7, 27]:
    syr_orb = syracuse_orbit(n)
    alpha_seq = alpha_sequence(n)
    tst = total_stopping_time(n)
    identity_value = len(alpha_seq) + sum(alpha_seq)
    
    print(f"n = {n}")
    print(f"  Syracuse orbit:  {syr_orb}")
    print(f"  Alpha sequence:  {alpha_seq}")
    print(f"  len(alpha_seq) = {len(alpha_seq)}, sum(alpha_seq) = {sum(alpha_seq)}")
    print(f"  total_stopping_time({n}) = {tst}")
    print(f"  Identity check: {len(alpha_seq)} + {sum(alpha_seq)} = {identity_value} == {tst}? {identity_value == tst}")
    assert identity_value == tst, f"Identity FAILED for n={n}"
    print()

## Alpha Values and Modular Arithmetic

The alpha value α(n) is determined by n's residue modulo powers of 2:
- n ≡ 3 (mod 4) → α = 1 (exactly)
- n ≡ 1 (mod 8) → α = 2 (exactly)
- n ≡ 5 (mod 8) → α ≥ 3

This is where **multiplicative structure meets binary structure**: for n = p × q, the residue n mod 2^k is determined by (p mod 2^k) × (q mod 2^k) mod 2^k.

In [ ]:
# Alpha vs residue classes for odd n from 3 to 999
rows = []
for n in range(3, 1000, 2):
    av = alpha_value(n)
    rows.append({
        'n': n,
        'alpha': av,
        'n_mod_4': n % 4,
        'n_mod_8': n % 8,
        'n_mod_16': n % 16,
    })

df_alpha = pd.DataFrame(rows)

# Crosstab: alpha distribution vs residue mod 8
print("Crosstab: alpha value vs n mod 8")
ct = pd.crosstab(df_alpha['n_mod_8'], df_alpha['alpha'], margins=True)
print(ct)
print()

# Verify: all n == 3 (mod 4) have alpha = 1
mod4_3 = df_alpha[df_alpha['n_mod_4'] == 3]
all_alpha_1 = (mod4_3['alpha'] == 1).all()
print(f"All n == 3 (mod 4) have alpha = 1? {all_alpha_1}")
assert all_alpha_1, "Failed: some n == 3 (mod 4) have alpha != 1"

# Verify: all n == 1 (mod 8) have alpha = 2
mod8_1 = df_alpha[df_alpha['n_mod_8'] == 1]
all_alpha_2 = (mod8_1['alpha'] == 2).all()
print(f"All n == 1 (mod 8) have alpha = 2? {all_alpha_2}")
assert all_alpha_2, "Failed: some n == 1 (mod 8) have alpha != 2"

# Verify: all n == 5 (mod 8) have alpha >= 3
mod8_5 = df_alpha[df_alpha['n_mod_8'] == 5]
all_alpha_ge3 = (mod8_5['alpha'] >= 3).all()
print(f"All n == 5 (mod 8) have alpha >= 3? {all_alpha_ge3}")
assert all_alpha_ge3, "Failed: some n == 5 (mod 8) have alpha < 3"

print("\nAlpha vs mod 8 relationship:")
for r in [1, 3, 5, 7]:
    subset = df_alpha[df_alpha['n_mod_8'] == r]
    alpha_dist = subset['alpha'].value_counts().sort_index()
    print(f"  n == {r} (mod 8): alpha distribution = {dict(alpha_dist)}")

## Factor Residues Determine First Alpha

For odd primes p, q: the product pq mod 4 is determined by p mod 4 and q mod 4.

| p mod 4 | q mod 4 | pq mod 4 | First α of pq |
|---------|---------|----------|----------------|
| 1       | 1       | 1        | ≥ 2            |
| 1       | 3       | 3        | = 1            |
| 3       | 3       | 1        | ≥ 2            |

This is the simplest example of multiplicative structure directly controlling Collatz dynamics.

In [ ]:
# Verify mod 4 prediction for all pairs of odd primes p, q < 100
odd_primes = [p for p in range(3, 100) if is_prime(p)]

correct_mod4 = 0
total_mod4 = 0
failures_mod4 = []

for i, p in enumerate(odd_primes):
    for q in odd_primes[i:]:
        pq = p * q
        av = alpha_value(pq)
        pq_mod4 = pq % 4
        
        # Prediction: if pq == 3 (mod 4) then alpha = 1; if pq == 1 (mod 4) then alpha >= 2
        if pq_mod4 == 3:
            predicted = (av == 1)
        else:  # pq_mod4 == 1
            predicted = (av >= 2)
        
        if predicted:
            correct_mod4 += 1
        else:
            failures_mod4.append((p, q, pq, av, pq_mod4))
        total_mod4 += 1

print(f"Mod 4 prediction accuracy: {correct_mod4}/{total_mod4} = {correct_mod4/total_mod4:.4f}")
if failures_mod4:
    print(f"Failures: {failures_mod4[:10]}")
else:
    print("No failures -- mod 4 prediction is exact.")

# Extend to mod 8: for pq == 1 (mod 8) -> alpha = 2 exactly
print("\n--- Mod 8 extension ---")
correct_mod8 = 0
total_mod8 = 0
mod8_details = Counter()

for i, p in enumerate(odd_primes):
    for q in odd_primes[i:]:
        pq = p * q
        av = alpha_value(pq)
        pq_mod8 = pq % 8
        mod8_details[(pq_mod8, av)] += 1
        
        if pq_mod8 == 3 or pq_mod8 == 7:  # == 3 mod 4
            predicted = (av == 1)
        elif pq_mod8 == 1:  # == 1 mod 8
            predicted = (av == 2)
        elif pq_mod8 == 5:  # == 5 mod 8
            predicted = (av >= 3)
        else:
            predicted = True  # shouldn't happen for odd products
        
        if predicted:
            correct_mod8 += 1
        total_mod8 += 1

print(f"Mod 8 prediction accuracy: {correct_mod8}/{total_mod8} = {correct_mod8/total_mod8:.4f}")

print("\nDistribution of (pq mod 8, alpha):")
for key in sorted(mod8_details.keys()):
    print(f"  pq == {key[0]} (mod 8), alpha = {key[1]}: count = {mod8_details[key]}")

# Extend to mod 16
print("\n--- Mod 16 extension ---")
mod16_details = Counter()
for i, p in enumerate(odd_primes):
    for q in odd_primes[i:]:
        pq = p * q
        av = alpha_value(pq)
        pq_mod16 = pq % 16
        mod16_details[(pq_mod16, av)] += 1

print("Distribution of (pq mod 16, alpha):")
for key in sorted(mod16_details.keys()):
    print(f"  pq == {key[0]:>2} (mod 16), alpha = {key[1]}: count = {mod16_details[key]}")

## Do Syracuse Orbits of Composites Pass Through Their Factors?

A striking question: when we compute the Syracuse orbit of n = p × q, does the orbit ever visit p or q?

In [ ]:
# Factor visitation analysis for odd semiprimes p*q up to 2000
odd_primes_list = [p for p in range(3, 1500) if is_prime(p)]

records = []
for i, p in enumerate(odd_primes_list):
    for q in odd_primes_list[i:]:
        pq = p * q
        if pq > 2000:
            break
        syr_orb = syracuse_orbit(pq)
        orbit_set = set(syr_orb)
        visits_p = p in orbit_set
        visits_q = q in orbit_set
        visits_either = visits_p or visits_q
        visits_both = visits_p and visits_q
        sc = stopping_class(pq)
        ratio = p / q if q != 0 else 0
        records.append({
            'p': p, 'q': q, 'pq': pq,
            'visits_smaller': visits_p,
            'visits_larger': visits_q,
            'visits_either': visits_either,
            'visits_both': visits_both,
            'stopping_class': sc,
            'ratio_p_q': ratio,
        })

df_visit = pd.DataFrame(records)

print(f"Total odd semiprimes p*q <= 2000 (p <= q): {len(df_visit)}")
print(f"\nVisitation summary:")
print(f"  Orbits visiting smaller factor (p): {df_visit['visits_smaller'].sum()} ({df_visit['visits_smaller'].mean():.3f})")
print(f"  Orbits visiting larger factor (q):  {df_visit['visits_larger'].sum()} ({df_visit['visits_larger'].mean():.3f})")
print(f"  Orbits visiting either factor:      {df_visit['visits_either'].sum()} ({df_visit['visits_either'].mean():.3f})")
print(f"  Orbits visiting both factors:        {df_visit['visits_both'].sum()} ({df_visit['visits_both'].mean():.3f})")

# Breakdown by ratio buckets
df_visit['ratio_bucket'] = pd.cut(df_visit['ratio_p_q'], bins=[0, 0.2, 0.4, 0.6, 0.8, 1.0], include_lowest=True)
print("\nVisitation rate by p/q ratio:")
ratio_summary = df_visit.groupby('ratio_bucket', observed=True)['visits_either'].agg(['mean', 'count'])
print(ratio_summary)

# Bar chart: fraction of orbits visiting a factor, grouped by stopping class
class_counts = df_visit.groupby('stopping_class')['visits_either'].agg(['mean', 'count'])
class_counts = class_counts[class_counts['count'] >= 3].sort_index()

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(range(len(class_counts)), class_counts['mean'], color='steelblue', edgecolor='black', alpha=0.8)
ax.set_xticks(range(len(class_counts)))
ax.set_xticklabels([str(c) for c in class_counts.index], rotation=45, ha='right', fontsize=8)
ax.set_xlabel('Stopping class of p*q')
ax.set_ylabel('Fraction of orbits visiting a factor')
ax.set_title('Factor Visitation Rate by Stopping Class of Semiprime')
ax.axhline(y=df_visit['visits_either'].mean(), color='red', linestyle='--', alpha=0.7,
           label=f'Overall rate: {df_visit["visits_either"].mean():.3f}')
ax.legend()
plt.tight_layout()
plt.show()

## Alpha Sequence Comparison: Composites vs Factors

If there's multiplicative structure in Syracuse dynamics, we might see it in how alpha sequences relate. For n = p × q, compare:
- alpha_sequence(n) vs alpha_sequence(p) and alpha_sequence(q)
- Do they share common subsequences? Common suffixes?

In [ ]:
# Alpha sequence comparison for semiprimes p*q < 500
def is_suffix(short, long):
    """Check if 'short' is a suffix of 'long'."""
    if len(short) > len(long):
        return False
    return long[-len(short):] == short


def longest_common_subsequence_length(a, b):
    """Length of longest common subsequence (not substring) of lists a, b."""
    m, n = len(a), len(b)
    if m == 0 or n == 0:
        return 0
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if a[i - 1] == b[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp[m][n]


primes_small = [p for p in range(3, 500) if is_prime(p)]
records_alpha = []

for i, p in enumerate(primes_small):
    for q in primes_small[i:]:
        pq = p * q
        if pq >= 500:
            break
        alpha_pq = alpha_sequence(pq)
        alpha_p = alpha_sequence(p)
        alpha_q = alpha_sequence(q)
        
        suffix_p = is_suffix(alpha_p, alpha_pq)
        suffix_q = is_suffix(alpha_q, alpha_pq)
        suffix_either = suffix_p or suffix_q
        
        lcs_p = longest_common_subsequence_length(alpha_pq, alpha_p)
        lcs_q = longest_common_subsequence_length(alpha_pq, alpha_q)
        
        records_alpha.append({
            'p': p, 'q': q, 'pq': pq,
            'alpha_pq': alpha_pq,
            'alpha_p': alpha_p,
            'alpha_q': alpha_q,
            'suffix_p': suffix_p,
            'suffix_q': suffix_q,
            'suffix_either': suffix_either,
            'lcs_with_p': lcs_p,
            'lcs_with_q': lcs_q,
        })

df_alpha_cmp = pd.DataFrame(records_alpha)

# Display first 20 examples
print("First 20 semiprimes and their alpha sequences:")
print(f"{'p':>3} {'q':>3} {'pq':>5}  alpha(pq){'':>20}  alpha(p){'':>12}  alpha(q){'':>12}  suffix?")
print("-" * 100)
for _, row in df_alpha_cmp.head(20).iterrows():
    apq_str = str(row['alpha_pq'])[:30]
    ap_str = str(row['alpha_p'])[:20]
    aq_str = str(row['alpha_q'])[:20]
    suf = "p" if row['suffix_p'] else ""
    suf += "q" if row['suffix_q'] else ""
    if not suf:
        suf = "-"
    print(f"{int(row['p']):>3} {int(row['q']):>3} {int(row['pq']):>5}  {apq_str:<30}  {ap_str:<20}  {aq_str:<20}  {suf}")

# Summary
total = len(df_alpha_cmp)
suffix_count = df_alpha_cmp['suffix_either'].sum()
print(f"\nSummary: {suffix_count}/{total} semiprimes ({suffix_count/total:.3f}) share a common suffix with a factor.")
print(f"  Suffix with smaller factor (p): {df_alpha_cmp['suffix_p'].sum()}")
print(f"  Suffix with larger factor (q):  {df_alpha_cmp['suffix_q'].sum()}")

# Average LCS overlap
print(f"\nAvg longest common subsequence length with p: {df_alpha_cmp['lcs_with_p'].mean():.2f}")
print(f"Avg longest common subsequence length with q: {df_alpha_cmp['lcs_with_q'].mean():.2f}")

## Residue Class Dynamics: Collatz mod 2^k

The Collatz map restricted to residues mod m is a finite dynamical system. Since alpha values are determined by residues mod 2^k, studying these finite systems reveals the algebraic skeleton of Collatz dynamics.

In [ ]:
# Residue dynamics of the Syracuse map mod 16 and mod 32

def syracuse_mod(r, mod, num_reps=20):
    """Compute S(r) mod `mod` using several representatives to check consistency."""
    results = set()
    for k in range(num_reps):
        n = r + mod * k
        if n < 1 or n % 2 == 0:
            continue
        s = syracuse_step(n)
        results.add(s % mod)
    return results


def build_transition_table(mod):
    """Build Syracuse transition table mod `mod` for odd residues."""
    odd_residues = [r for r in range(1, mod, 2)]
    table = {}
    for r in odd_residues:
        targets = syracuse_mod(r, mod, num_reps=50)
        table[r] = targets
    return table


def find_cycles(table):
    """Find cycles in a transition table (only for deterministic transitions)."""
    deterministic = {r: next(iter(targets)) for r, targets in table.items() if len(targets) == 1}
    cycles = []
    visited_global = set()
    for start in deterministic:
        if start in visited_global:
            continue
        path = []
        current = start
        visited = set()
        while current in deterministic and current not in visited:
            visited.add(current)
            path.append(current)
            current = deterministic[current]
        if current in visited:
            cycle_start = path.index(current)
            cycle = path[cycle_start:]
            if tuple(sorted(cycle)) not in [tuple(sorted(c)) for c in cycles]:
                cycles.append(cycle)
            visited_global.update(cycle)
    return cycles


# --- Mod 16 ---
print("=" * 60)
print("Syracuse map mod 16 (odd residues)")
print("=" * 60)
table_16 = build_transition_table(16)
for r in sorted(table_16):
    targets = table_16[r]
    det = "deterministic" if len(targets) == 1 else "non-deterministic"
    print(f"  S({r:>2}) mod 16 -> {sorted(targets)}  ({det})")

cycles_16 = find_cycles(table_16)
print(f"\nCycles in mod-16 dynamics: {cycles_16}")

# --- Mod 32 ---
print("\n" + "=" * 60)
print("Syracuse map mod 32 (odd residues)")
print("=" * 60)
table_32 = build_transition_table(32)
for r in sorted(table_32):
    targets = table_32[r]
    det = "deterministic" if len(targets) == 1 else "non-deterministic"
    print(f"  S({r:>2}) mod 32 -> {sorted(targets)}  ({det})")

cycles_32 = find_cycles(table_32)
print(f"\nCycles in mod-32 dynamics: {cycles_32}")

# Visualize mod 16 as a directed graph
fig, ax = plt.subplots(figsize=(8, 8))
odd_res_16 = [r for r in range(1, 16, 2)]
n_nodes = len(odd_res_16)
angles = {r: 2 * np.pi * i / n_nodes for i, r in enumerate(odd_res_16)}
radius = 3.0
positions = {r: (radius * np.cos(angles[r]), radius * np.sin(angles[r])) for r in odd_res_16}

# Draw nodes
for r in odd_res_16:
    x, y = positions[r]
    circle = plt.Circle((x, y), 0.35, color='lightblue', ec='black', linewidth=1.5, zorder=3)
    ax.add_patch(circle)
    ax.text(x, y, str(r), ha='center', va='center', fontsize=11, fontweight='bold', zorder=4)

# Draw edges
for r in odd_res_16:
    for t in table_16[r]:
        if t in positions:
            x0, y0 = positions[r]
            x1, y1 = positions[t]
            dx, dy = x1 - x0, y1 - y0
            dist = np.sqrt(dx**2 + dy**2)
            if dist > 0:
                # Shorten arrow to avoid overlapping with node circles
                shrink = 0.4 / dist
                ax.annotate("",
                            xy=(x1 - dx * shrink, y1 - dy * shrink),
                            xytext=(x0 + dx * shrink, y0 + dy * shrink),
                            arrowprops=dict(arrowstyle='->', color='darkblue',
                                            lw=1.5, connectionstyle='arc3,rad=0.15'),
                            zorder=2)

ax.set_xlim(-4.5, 4.5)
ax.set_ylim(-4.5, 4.5)
ax.set_aspect('equal')
ax.set_title('Syracuse Map mod 16 (Odd Residues)', fontsize=14)
ax.axis('off')
plt.tight_layout()
plt.show()

## 2-adic Valuation Profiles by Factorization Type

For each number, the distribution of alpha values (how many 1s, 2s, 3s, 4s, etc.) provides a "profile" of the orbit. Do primes, semiprimes, and prime powers have distinguishable profiles?

In [ ]:
# Alpha profiles for odd numbers 3 to 5000
def classify_number(n):
    """Classify n as prime, semiprime, prime_power, or other_composite."""
    if is_prime(n):
        return 'prime'
    factors = prime_factorization(n)
    total_exp = sum(factors.values())
    num_distinct = len(factors)
    if num_distinct == 1:
        return 'prime_power'
    if total_exp == 2:
        return 'semiprime'
    return 'other_composite'


profile_rows = []
for n in range(3, 5001, 2):
    ntype = classify_number(n)
    aseq = alpha_sequence(n)
    if len(aseq) == 0:
        continue  # n=1 edge case
    alpha_counts = Counter(aseq)
    mean_alpha = np.mean(aseq)
    var_alpha = np.var(aseq) if len(aseq) > 1 else 0.0
    profile_rows.append({
        'n': n,
        'type': ntype,
        'alpha_len': len(aseq),
        'alpha_sum': sum(aseq),
        'mean_alpha': mean_alpha,
        'var_alpha': var_alpha,
        'frac_alpha_1': alpha_counts.get(1, 0) / len(aseq),
        'frac_alpha_2': alpha_counts.get(2, 0) / len(aseq),
        'max_alpha': max(aseq),
    })

df_profiles = pd.DataFrame(profile_rows)

# Summary statistics
print("Mean alpha by number type:")
summary = df_profiles.groupby('type')['mean_alpha'].agg(['mean', 'std', 'count'])
print(summary)
print()

print("Mean alpha sequence length by number type:")
print(df_profiles.groupby('type')['alpha_len'].agg(['mean', 'std']))

# Boxplots of mean alpha grouped by number type
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

type_order = ['prime', 'semiprime', 'prime_power', 'other_composite']
type_labels = ['Prime', 'Semiprime', 'Prime Power', 'Other Composite']

# Boxplot of mean alpha
data_box = [df_profiles[df_profiles['type'] == t]['mean_alpha'].values for t in type_order]
bp = axes[0].boxplot(data_box, labels=type_labels, patch_artist=True)
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[0].set_ylabel('Mean alpha value')
axes[0].set_title('Distribution of Mean Alpha by Number Type')
axes[0].tick_params(axis='x', rotation=20)

# Scatter plot of (mean_alpha, len_alpha_sequence) colored by type
for t, color, label in zip(type_order, colors, type_labels):
    subset = df_profiles[df_profiles['type'] == t]
    axes[1].scatter(subset['mean_alpha'], subset['alpha_len'],
                    c=color, alpha=0.3, s=8, label=label)
axes[1].set_xlabel('Mean alpha value')
axes[1].set_ylabel('Alpha sequence length (Syracuse steps)')
axes[1].set_title('Alpha Profile: Mean vs Length')
axes[1].legend(markerscale=3, fontsize=8)

plt.tight_layout()
plt.show()

## The Syracuse "Multiplication Table"

For odd primes p, q, define the "Syracuse class" as the dropping_set (= stopping class). Is class(p × q) a function of (class(p), class(q))?

In [ ]:
# Syracuse multiplication table for odd primes p, q < 80
primes_80 = [p for p in range(3, 80) if is_prime(p)]

mult_rows = []
for p in primes_80:
    for q in primes_80:
        pq = p * q
        class_p = dropping_set(p)
        class_q = dropping_set(q)
        class_pq = dropping_set(pq)
        mult_rows.append({
            'p': p, 'q': q, 'pq': pq,
            'class_p': class_p,
            'class_q': class_q,
            'class_pq': class_pq,
        })

df_mult = pd.DataFrame(mult_rows)

# For each (class_p, class_q), how many distinct class_pq values?
consistency = df_mult.groupby(['class_p', 'class_q'])['class_pq'].agg(
    ['nunique', lambda x: x.value_counts().index[0], 'count']
)
consistency.columns = ['n_distinct', 'most_common_class_pq', 'count']
consistency = consistency.reset_index()

print("Consistency check: for fixed (class_p, class_q), how many distinct class(pq)?")
print(consistency.to_string(index=False))

perfectly_consistent = (consistency['n_distinct'] == 1).all()
print(f"\nPerfectly consistent (class(pq) is a function of class(p), class(q))? {perfectly_consistent}")

if not perfectly_consistent:
    avg_distinct = consistency['n_distinct'].mean()
    max_distinct = consistency['n_distinct'].max()
    print(f"Average distinct values: {avg_distinct:.2f}, Max: {max_distinct}")

# Pivot table: most common class(pq) for each (class_p, class_q)
pivot = consistency.pivot(index='class_p', columns='class_q', values='most_common_class_pq')

# Heatmap visualization
fig, ax = plt.subplots(figsize=(10, 8))
pivot_vals = pivot.values.astype(float)
im = ax.imshow(pivot_vals, cmap='viridis', aspect='auto')

# Annotate cells
for i in range(pivot_vals.shape[0]):
    for j in range(pivot_vals.shape[1]):
        val = pivot_vals[i, j]
        if not np.isnan(val):
            ax.text(j, i, f"{int(val)}", ha='center', va='center',
                    color='white' if val > np.nanmedian(pivot_vals) else 'black',
                    fontsize=9)

ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([str(c) for c in pivot.columns])
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([str(c) for c in pivot.index])
ax.set_xlabel('class(q) = dropping_set(q)')
ax.set_ylabel('class(p) = dropping_set(p)')
ax.set_title('Most Common class(p*q) for Each (class(p), class(q))')
plt.colorbar(im, ax=ax, label='Most common class(pq)')
plt.tight_layout()
plt.show()

# Also show the number of distinct classes per pair
pivot_nunique = consistency.pivot(index='class_p', columns='class_q', values='n_distinct')
print("\nNumber of distinct class(pq) values per (class_p, class_q) pair:")
print(pivot_nunique.to_string())

## Key Findings

*To be filled after running the notebook.*

### Confirmed
- Alpha values are fully determined by residues mod 2^k
- Factor residue classes (mod 4) predict the first alpha of their product
- Total stopping time decomposes exactly into odd steps + even steps via alpha sequences

### Open Questions
- Is the factor-visitation rate (orbits passing through factors) explainable, or is it just base-rate?
- Do alpha sequence suffixes carry factorization information?
- Can we predict stopping class of pq from classes of p and q?